# SI100B Fall 2026 Recitation 5 (A)

## Lec09–10 Review Workbook

配套 `Recitation5_A.md` 使用。先读场景、预测输出，再运行代码；每节都有可直接运行的例子。

- **1–7** 按讲义顺序回顾知识点，每个例子旁都说明要观察什么。
- **8 Exercises** 只给题目和动手位置。先在此完成，再对照 **9 Check Your Answers**。
- **10 Reference Solutions** 提供参考实现和检查，不需要先写练习也能从头运行。

使用 Python 3，从上往下运行即可；也可以选择 **Restart Kernel and Run All Cells**。
只用 Python 标准库，没有交互式 `input()`、外部数据或额外安装要求。需要故意触发的异常会在演示中捕获并显示，整本运行不会因此中断。

课件依据为 `class/Lec09_A.pdf` 和 `class/Lec10_A.pdf`。页码采用 PDF 页码。
重复内容合并复习；标签文本和打印费用等场景为本习题课自造示例。


| 小节 | 对照课件 | 主要场景 |
|:---|:---|:---|
| 1 List Comprehension | Lec09 pp. 2–4 | 整理成绩与标签 |
| 2 Functions Returning Functions | Lec09 pp. 5–23 | 为不同打印机生成计价函数 |
| 3 Testing and Debugging | Lec09 pp. 24–38 | 查找平均分与绝对值中的错误 |
| 4 Exceptions and Assertions | Lec09 pp. 39–60；Lec10 pp. 3–10 | 分摊费用、汇总成绩 |
| 5 Dictionaries | Lec09 pp. 61–87；Lec10 pp. 11–37 | 名单、座位与嵌套成绩表 |
| 6 Word Frequency | Lec10 pp. 38–51 | 社团兴趣标签统计 |
| 7 Recursion | Lec10 pp. 52–88 | 糖果、幂、排队与调用追踪 |


## 1. List Comprehension

### 1.1 场景：保留及格成绩，再加 5 分

先用普通循环，再改写成列表推导式。比较两者的输出，并检查原列表是否改变。
读推导式时，先找 `for`，再找筛选条件 `if`，最后看左边要生成的值。


In [1]:
scores = [48, 60, 75, 92]
adjusted_loop = []
for score in scores:
    if score >= 60:
        adjusted_loop.append(score + 5)

adjusted = [score + 5 for score in scores if score >= 60]
squares = [number**2 for number in range(4)]
print("循环结果", adjusted_loop)  # [65, 80, 97]
print("推导式结果", adjusted)  # [65, 80, 97]
print("原成绩", scores)  # [48, 60, 75, 92]
print("不需要筛选时", squares)  # [0, 1, 4, 9]
assert adjusted == adjusted_loop == [65, 80, 97]
assert scores == [48, 60, 75, 92]

循环结果 [65, 80, 97]
推导式结果 [65, 80, 97]
原成绩 [48, 60, 75, 92]
不需要筛选时 [0, 1, 4, 9]


### 1.2 场景：混合数据中只统计字符串长度

这是 Lec09 的追踪题。对每个元素，先筛选，再执行 `len(x)`。
因此整数 `7` 被跳过，不会执行 `len(7)`；带引号的 `'4.0'` 是长度为 3 的字符串。


In [2]:
data = ["xy", "abcd", 7, "4.0"]
lengths = [len(x) for x in data if type(x) == str]
print(lengths)  # [2, 4, 3]
print([len(x) for x in []])  # []，没有元素就不执行表达式
assert lengths == [2, 4, 3]

[2, 4, 3]
[]


## 2. Functions Returning Functions

### 2.1 场景：黑白打印和彩色打印

两种打印机共用「单价 × 页数」规则。`make_cost(unit_price)` 返回一个计价函数，
调用计价函数时再提供页数。注意 `return cost` 没有调用括号。


In [ ]:
def make_cost(unit_price):
    def cost(pages):
        return unit_price * pages

    return cost


black_white = make_cost(2)
color = make_cost(5)
print("返回的是函数吗？", callable(black_white))  # True
print("黑白 3 页", black_white(3))  # 6
print("彩色 3 页", color(3))  # 15
print("连续两次调用", make_cost(2)(3))  # 6
assert black_white(3) == 6
assert color(3) == 15

### 2.2 外层调用结束后，计价规则仍然可用

`pages` 在本次内部调用的局部作用域中；`unit_price` 在定义内部函数时的外层作用域中。
内部函数仍能访问所需的外层绑定，这就是闭包（closure）。
不同的 `make_cost` 调用产生不同的绑定；这里不是把单价放进了全局变量。


In [ ]:
cheap = make_cost(2)
expensive = make_cost(5)
for pages in [0, 1, 4]:
    print(pages, "页：", cheap(pages), expensive(pages))
# 0 页：0 0；1 页：2 5；4 页：8 20
assert cheap(4) == 8
assert expensive(4) == 20

### 2.3 函数对象和调用结果

`make_prod(2)` 返回函数；`make_prod(2)(3)` 返回数字。
第二个版本直接返回 `g(3)`，因此拿到的是已经算好的结果。
这两个接口用途不同，不能把数字当成函数继续调用。


In [ ]:
def make_prod(a):
    def g(b):
        return a * b

    return g


def product_with_three(a):
    def g(b):
        return a * b

    return g(3)


doubler = make_prod(2)
ready_value = product_with_three(2)
print(callable(doubler), doubler(3))  # True 6
print(callable(ready_value), ready_value)  # False 6
assert make_prod(2)(3) == 6

## 3. Testing and Debugging

### 3.1 先写规格，再写期望结果

场景是计算平均成绩。约定 `grades` 是非空数值列表，返回算术平均值，不修改输入。
这些用例属于从规格出发的黑盒测试，分别覆盖单项、端点和多项。
非空约定的失败情况会在第 4 节演示。


In [ ]:
def average(grades):
    """grades 是非空数值列表；返回算术平均值，不修改输入。"""
    assert len(grades) > 0, "no grades data"
    return sum(grades) / len(grades)


average_cases = [([80], 80.0), ([0, 100], 50.0), ([60, 80, 100], 80.0)]
for grades, expected in average_cases:
    before = grades.copy()
    actual = average(grades)
    print(grades, "->", actual, "期望", expected)
    assert actual == expected
    assert grades == before

### 3.2 白盒测试的陷阱：分支都测过，边界仍可能有错

下面故意写错绝对值的条件。`-2` 与 `2` 能走过两个分支，但不能揭示 `-1` 的错误。
先观察输出，再将条件改正，保留失败过的输入做回归测试。
测试通过只能说明这些用例通过，不能证明程序对所有输入都正确。


In [ ]:
def absolute_buggy(x):
    if x < -1:
        return -x
    return x


for x in [-2, 2, -1, 0]:
    print("输入", x, "错误版本", absolute_buggy(x), "正确结果", abs(x))
assert absolute_buggy(-1) == -1  # 确认反例，避免让演示单元格中断


def absolute_fixed(x):
    if x < 0:
        return -x
    return x


for x in [-2, -1, 0, 1, 2]:
    assert absolute_fixed(x) == abs(x)
print("修正后，边界与回归用例通过")

### 3.3 场景：平均分为何总是偏小？

调试顺序是「发现偏差 → 构造最小反例 → 提出猜想 → 打印验证 → 修复 → 重测」。
这里用两项成绩 `[60, 100]`，期望 `80.0`。留意循环实际访问了哪些下标。


In [ ]:
def average_buggy(grades):
    total = 0
    for i in range(len(grades) - 1):
        total += grades[i]
        print("访问下标", i, "当前总分", total)
    return total / len(grades)


print("错误结果", average_buggy([60, 100]))  # 30.0，循环只访问下标 0

`range(len(grades) - 1)` 的右端点不包含在内，最后一项被漏掉。
改成 `range(len(grades))` 后，重新检查单项和多项输入。
这里保留循环形式，是为了让修改与原因直接对应。


In [ ]:
def average_fixed(grades):
    assert len(grades) > 0, "no grades data"
    total = 0
    for i in range(len(grades)):
        total += grades[i]
    return total / len(grades)


for grades, expected in [([80], 80.0), ([60, 100], 80.0), ([0, 100], 50.0)]:
    assert average_fixed(grades) == expected
print("修复后", average_fixed([60, 100]))  # 80.0

### 3.4 单元、集成、回归，以及循环次数

- 单元测试检查一个函数。
- 集成测试检查函数之间的配合。
- 回归测试保留旧错误对应的用例，防止修改后复发。

测试循环时，覆盖 0 次、1 次、多次。下面空列表的总分按约定为 `0`，与空列表不能求平均并不冲突。


In [ ]:
def total_score(grades):
    total = 0
    for grade in grades:
        total += grade
    return total


for grades, expected in [([], 0), ([80], 80), ([60, 100], 160)]:
    assert total_score(grades) == expected
    print(grades, "总分", total_score(grades))

## 4. Exceptions and Assertions

### 4.1 场景：分摊一次活动费用

用变量模拟输入，避免 notebook 等待键盘。
`'12' / '3'` 对应正常计算；非整数字符串触发 `ValueError`；人数为零触发 `ZeroDivisionError`。
这里的 `try/except` 是本节要学习的异常处理，不会用无类型的 `except` 吞掉其他问题。


In [ ]:
for total_text, people_text in [("12", "3"), ("12", "three"), ("12", "0")]:
    print("本次输入", total_text, people_text)
    try:
        total = int(total_text)
        people = int(people_text)
        print("每人费用", total / people)
    except ValueError:
        print("请输入整数")
    except ZeroDivisionError:
        print("人数不能为 0")
# 依次得到：4.0；请输入整数；人数不能为 0

### 4.2 异常会跳过 try 中剩下的语句

先预测字母输出顺序。处理完成后，程序从整个 `try/except` 后面继续，
不会回到报错位置重试，也不会执行被跳过的 `B`。


In [ ]:
try:
    print("A")
    value = int("oops")
    print("B")
except ValueError:
    print("C")
print("D")
# 输出 A、C、D

### 4.3 常见异常与报错信息

这里只在三个独立的小例子中捕获预期类型，显示「类型 + 信息」。
实际排查时，还应阅读 traceback 中自己代码的文件位置和行号。
另两个常见类型是 `TypeError`（例如字符串加整数）和 `NameError`（使用未定义名字）。


In [ ]:
try:
    print([80][1])  # noqa: PLE0643 -- 故意演示并捕获下标越界异常
except IndexError as error:
    print(type(error).__name__, str(error))

try:
    print({"Ana": 80}["Bob"])
except KeyError as error:
    print(type(error).__name__, str(error))

try:
    print("80" + 5)
except TypeError as error:
    print(type(error).__name__, str(error))

### 4.4 else 与 finally（了解）

`else` 只在 `try` 正常完成时执行；正常 Python 控制流离开结构时会执行 `finally`。
本例中正常路径为「转换成功 → 本次处理结束」，失败路径为「转换失败 → 本次处理结束」。
`else` 中产生的新异常不由同一结构的 `except` 捕获。


In [ ]:
for text in ["12", "oops"]:
    print("输入", text)
    try:
        value = int(text)
    except ValueError:
        print("转换失败")
    else:
        print("转换成功", value)
    finally:
        print("本次处理结束")

### 4.5 raise：将底层错误变成接口约定的错误

场景是读取活动人数。接口约定只接受正整数形式的文本，非法值应抛出 `ValueError`。
`raise` 中断当前函数；这里由调用者捕获，以便继续展示下一种输入。


In [ ]:
def read_people(text):
    people = int(text)
    if people <= 0:
        raise ValueError("人数必须大于 0")
    return people


assert read_people("3") == 3
for text in ["zero", "0", "-2"]:
    try:
        read_people(text)
    except ValueError as error:
        print(repr(text), "->", type(error).__name__, str(error))
    else:
        raise AssertionError("这些输入应当抛出 ValueError")

### 4.6 assert：检查本来就应该成立的约定

沿用第 3 节的 `average`。空列表违反非空约定，触发 `AssertionError`。
被调用函数的计算在断言处中断；这里调用者捕获后，notebook 可以继续。
断言在 Python `-O` 模式下可被移除，因此不应用它代替必须执行的外部输入校验。


In [ ]:
print(average([80, 100]))  # 90.0
try:
    average([])
except AssertionError as error:
    print(type(error).__name__, str(error))  # AssertionError no grades data
else:
    raise AssertionError("空列表应当触发断言")

### 4.7 没成绩时的三种政策

老师用空成绩列表展示三种选择。下面显式写出 `return None`，便于观察。
把缺少成绩当成 `0.0` 会改变后续统计，不能在未说明规则时偷偷替换。


In [ ]:
def average_or_none(grades):
    try:
        return sum(grades) / len(grades)
    except ZeroDivisionError:
        print("warning: no grades data")
        return None


def average_or_zero(grades):
    try:
        return sum(grades) / len(grades)
    except ZeroDivisionError:
        print("warning: no grades data")
        return 0.0


print("政策一", average_or_none([]))  # None
print("政策二", average_or_zero([]))  # 0.0
print("政策三见上一个单元格：average([]) 抛 AssertionError")

### 4.8 集成测试：将平均分加进学生记录

记录结构沿用正课的「姓名列表、成绩列表」。这里选择严格的非空成绩约定。
`average` 已有单元测试；现在再检查 `get_stats` 调用它之后的完整结果。
这个函数新建外层结果，但没有深拷贝姓名和成绩列表。


In [ ]:
def get_stats(class_list):
    new_stats = []
    for student in class_list:
        new_stats.append([student[0], student[1], average(student[1])])
    return new_stats


class_list = [[["Ana", "Li"], [60, 80]], [["Bob", "Wu"], [90]]]
statistics = get_stats(class_list)
print(statistics)
# [[['Ana', 'Li'], [60, 80], 70.0], [['Bob', 'Wu'], [90], 90.0]]
assert statistics == [
    [["Ana", "Li"], [60, 80], 70.0],
    [["Bob", "Wu"], [90], 90.0],
]
assert len(class_list[0]) == 2  # 原记录没有被添加平均分字段

## 5. Dictionaries

### 5.1 场景：姓名到成绩的对应关系

平行列表要保持下标一致；字典直接用姓名查询。
键不存在时会抛出 `KeyError`，第 4 节已经展示。


In [ ]:
names = ["Ana", "Bob", "Chen"]
scores = [80, 95, 72]
print("平行列表", scores[names.index("Bob")])  # 95

grades = {"Ana": 80, "Bob": 95, "Chen": 72}
print("字典查询", grades["Bob"])  # 95
empty_dictionary = {}
assert len(empty_dictionary) == 0

### 5.2 查、增、改、删与重复键

键不存在时赋值是新增，键已存在时赋值是更新。
同一个键只能保留当前的一个值。


In [ ]:
grades = {"Ana": 80, "Bob": 95}
print(grades["Ana"])  # 80
grades["Chen"] = 72
grades["Ana"] = 88
del grades["Bob"]
print(grades)  # {'Ana': 88, 'Chen': 72}
assert grades == {"Ana": 88, "Chen": 72}

menu = {}
menu["tea"] = 5
menu["tea"] = 6
print(menu, len(menu))  # {'tea': 6} 1

### 5.3 in 检查键；values() 才能检查值

`keys()`、`values()`、`items()` 返回可迭代视图，按需转成列表。
同时需要姓名和成绩时，用 `for name, score in grades.items()` 解包。


In [ ]:
grades = {"Ana": 80, "Bob": 95}
print("Ana" in grades)  # True
print(80 in grades)  # False
print(80 in grades.values())  # True
print(list(grades.keys()))  # ['Ana', 'Bob']
print(list(grades.values()))  # [80, 95]
print(list(grades.items()))  # [('Ana', 80), ('Bob', 95)]
for name, score in grades.items():
    print(name, "got", score)

### 5.4 插入顺序不等于排序

Python 3.7 起，字典保证插入顺序。修改已有键的值不改变其位置；删掉再插入会放到末尾。
老师课件用「不假设顺序」提醒大家按键查询；需要排序输出时仍要显式 `sorted(...)`。


In [ ]:
grades = {"Bob": 95, "Ana": 80}
grades["Bob"] = 96
print("插入顺序", list(grades))  # ['Bob', 'Ana']
print("按名字排序", sorted(grades))  # ['Ana', 'Bob']
del grades["Bob"]
grades["Bob"] = 96
print("删除再插入", list(grades))  # ['Ana', 'Bob']

### 5.5 场景：用楼排与座位号组成键

键要求可哈希。只含可哈希元素的元组可以作为键，含列表的元组不行。
字典用哈希值帮助定位，再比较键；不同键可能哈希冲突，字典会处理。
课件的「字母和取余」是模型，不是 Python 实际字符串哈希算法。


In [ ]:
seats = {("A", 1): "Ana", ("A", 2): "Bob"}
print(seats[("A", 2)])  # Bob

try:
    invalid_seats = {("A", [1, 2]): "Group"}
except TypeError as error:
    print(type(error).__name__, str(error))  # unhashable type: 'list'

### 5.6 别名与浅拷贝

`alias` 与 `original` 是同一个字典。`shallow` 是新的外层字典，但里面的列表仍然共享。
先预测两个 `print` 的结果，再运行。


In [1]:
original = {"Ana": [80, 90]}
alias = original
shallow = original.copy()
alias["Bob"] = [70]
shallow["Ana"].append(100)
print(original)  # {'Ana': [80, 90, 100], 'Bob': [70]}
print(shallow)  # {'Ana': [80, 90, 100]}
assert alias is original
assert shallow is not original
assert shallow["Ana"] is original["Ana"]

{'Ana': [80, 90, 100], 'Bob': [70]}
{'Ana': [80, 90, 100]}


### 5.7 连嵌套内容一起复制

需要独立修改内部列表时，用标准库的 `copy.deepcopy`。
这里的副本添加成绩后，原数据保持 `[80, 90]`。


In [ ]:
import copy

original = {"Ana": [80, 90]}
deep = copy.deepcopy(original)
deep["Ana"].append(100)
print("原数据", original)  # {'Ana': [80, 90]}
print("深拷贝", deep)  # {'Ana': [80, 90, 100]}
assert original["Ana"] == [80, 90]

### 5.8 嵌套字典：每次方括号走一层

`students['Ana']` 得到 Ana 的字典，`['mq']` 得到她的小测列表，`[0]` 得到第一次分数。


In [ ]:
students = {
    "Ana": {"mq": [8, 10], "ps": [9]},
    "Bob": {"mq": [6], "ps": [7, 8]},
}
print(students["Ana"])  # {'mq': [8, 10], 'ps': [9]}
print(students["Ana"]["mq"])  # [8, 10]
print(students["Ana"]["mq"][0])  # 8

### 5.9 extend 与 append 得到不同的结构

汇总所有小测成绩需要把每个分数放进同一个列表。
`extend` 添加一份列表中的各个元素，`append` 把整份列表当成一个元素。
`all_scores = all_scores + record['mq']` 也能完成这里的汇总。


In [ ]:
flat = []
nested = []
for record in students.values():
    flat.extend(record["mq"])
    nested.append(record["mq"])
print("extend", flat)  # [8, 10, 6]
print("append", nested)  # [[8, 10], [6]]
assert flat == [8, 10, 6]
assert nested == [[8, 10], [6]]

### 5.10 汇总成绩的平均，不是学生平均分的平均

所有成绩的平均为 $(8+10+6)/3=8$；先求每个人的平均再平均为 $(9+6)/2=7.5$。
当每个人的成绩数量不同，两种权重不同。正课题目要求前一种。
约定每条学生记录都包含 `what` 对应的列表，汇总后至少有一条成绩。


In [ ]:
def get_average(data, what):
    all_scores = []
    for record in data.values():
        all_scores.extend(record[what])
    assert len(all_scores) > 0, "no grades data"
    return sum(all_scores) / len(all_scores)


print("所有小测分数的平均", get_average(students, "mq"))  # 8.0
student_means = [average(record["mq"]) for record in students.values()]
print("每人平均的平均", average(student_means))  # 7.5
assert get_average(students, "mq") == 8.0

## 6. Word Frequency

### 6.1 场景：统计社团报名的兴趣标签

下面是自造文本，只做大小写统一和空白分词，不处理标点。
返回「标签字符串 → 次数整数」的字典。第一次出现设为 1，以后出现累加。


In [ ]:
def generate_word_dict(text):
    counts = {}
    for word in text.lower().split():
        if word in counts:
            counts[word] += 1
        else:
            counts[word] = 1
    return counts


text = "Python art python music ART python music"
word_counts = generate_word_dict(text)
print(word_counts)  # {'python': 3, 'art': 2, 'music': 2}
assert word_counts == {"python": 3, "art": 2, "music": 2}
assert generate_word_dict("   ") == {}

### 6.2 并列第一也要保留

`max(counts.values())` 找到最高频率，再遍历收集同频词。
函数返回 `(词列表, 最高频率)`，前提是输入字典非空，频率为正整数。


In [ ]:
def find_frequent_word(counts):
    assert len(counts) > 0, "empty frequency dictionary"
    highest = max(counts.values())
    words = []
    for word, frequency in counts.items():
        if frequency == highest:
            words.append(word)
    return words, highest


print(find_frequent_word(word_counts))  # (['python'], 3)
print(find_frequent_word({"art": 2, "music": 2}))  # (['art', 'music'], 2)
assert find_frequent_word({"art": 2, "music": 2}) == (["art", "music"], 2)

### 6.3 找出次数严格大于阈值的频率组

按照老师「找最高频 → 删除这一组 → 重复」的流程。
本版本先复制输入，避免调用后原词频丢失；先检查 `remaining`，避免删空后对空序列求 `max`。
删除时遍历的是独立的 `words` 列表，没有边遍历字典边改变它的大小。


In [ ]:
def occurs_often(counts, x):
    remaining = counts.copy()
    groups = []
    while remaining:
        words, highest = find_frequent_word(remaining)
        if highest <= x:
            break
        groups.append((words, highest))
        for word in words:
            del remaining[word]
    return groups


before = word_counts.copy()
print(occurs_often(word_counts, 1))
# [(['python'], 3), (['art', 'music'], 2)]
print("输入仍然保留", word_counts)
assert word_counts == before

### 6.4 从边界条件设计测试

检查空字典、阈值恰好相等、所有词都被选中、只有一词。
并列词在本实现中按字典插入顺序排列；频率组按频率递减排列。
输入频率来自前面的计数函数，均为正整数。


In [ ]:
assert occurs_often({}, 1) == []
assert occurs_often({"python": 3}, 3) == []
assert occurs_often({"python": 3}, 0) == [(["python"], 3)]
assert occurs_often({"art": 2, "music": 2}, 0) == [(["art", "music"], 2)]
assert occurs_often(word_counts, 2) == [(["python"], 3)]
assert occurs_often(word_counts, 0) == [(["python"], 3), (["art", "music"], 2)]
print("词频分组的边界检查通过")

## 7. Recursion

### 7.1 场景：每袋 a 颗糖，b 袋有多少颗？

迭代用变量 `result` 保存当前总数；递归改问「一袋的糖 + 剩余袋数的糖」。
约定 `b` 为非负整数。本讲用 `b == 0` 作为基例，扩展正课 `b >= 1` 的版本。


In [ ]:
def mult_iter(a, b):
    assert type(b) == int and b >= 0
    result = 0
    for _ in range(b):
        result += a
    return result


def mult_recur(a, b):
    assert type(b) == int and b >= 0
    if b == 0:
        return 0
    return a + mult_recur(a, b - 1)


for bags in [0, 1, 4]:
    iterative = mult_iter(5, bags)
    recursive = mult_recur(5, bags)
    print(bags, "袋：", iterative, recursive)  # 0、5、20
    assert iterative == recursive == 5 * bags

### 7.2 基例、递推式与进展

- 基例为 $M(a,0)=0$，直接返回。
- 递推式为 $M(a,b)=a+M(a,b-1)$。
- `b` 从非负整数出发，每次减一，因此到达零。

下面在进入与返回时分别打印。缩进表示深度，`depth` 是明确传入的演示参数。
看清「先向下调用，后向上返回」，不要只追踪下降的一半。


In [ ]:
def mult_trace(a, b, depth):
    assert type(b) == int and b >= 0
    indent = "  " * depth
    print(f"{indent}进入 mult({a}, {b})")
    if b == 0:
        print(f"{indent}返回 0")
        return 0
    smaller = mult_trace(a, b - 1, depth + 1)
    result = a + smaller
    print(f"{indent}返回 {a} + {smaller} = {result}")
    return result


assert mult_trace(5, 3, 0) == 15

### 7.3 场景：每轮数量翻倍，p 轮后是多少？

用幂函数演示递归。`p` 是非负整数；本练习将零次幂定义为 `1`，包括 `power_recur(0, 0)`。
基例 $n^0=1$；递推式 $n^p=n	imes n^{p-1}$。
不需要单独写 `p == 1`，因为它会递归到零次幂。


In [ ]:
def power_recur(n, p):
    assert type(p) == int and p >= 0
    if p == 0:
        return 1
    return n * power_recur(n, p - 1)


for base, exponent in [(2, 0), (2, 3), (-2, 3), (0, 3), (0, 0)]:
    result = power_recur(base, exponent)
    print(base, "的", exponent, "次幂 =", result)
    assert result == base**exponent

### 7.4 场景：n 位同学排队

第一个位置有 `n` 种选择，剩下 `n-1` 人的排列数是同类问题。
递推式为 $n!=n	imes(n-1)!$，基例为 $0!=1$，表示空排列的一种方式。
这里用零作为基例，以便同时支持 `n == 0`。


In [ ]:
def factorial(n):
    assert type(n) == int and n >= 0
    if n == 0:
        return 1
    return n * factorial(n - 1)


for n, expected in [(0, 1), (1, 1), (3, 6), (4, 24)]:
    print(n, "人排队：", factorial(n), "种")
    assert factorial(n) == expected

### 7.5 每层有自己的局部 n

下一层的 `n` 不会覆盖上一层。观察每次「准备返回」时的 `n`，
它仍然是该层进入时的值。局部名字独立不代表所有对象都被复制，传入可变对象时仍可共享。


In [ ]:
def factorial_trace(n, depth):
    assert type(n) == int and n >= 0
    indent = "  " * depth
    print(f"{indent}进入：本层 n = {n}")
    if n == 0:
        print(f"{indent}基例返回 1")
        return 1
    smaller = factorial_trace(n - 1, depth + 1)
    result = n * smaller
    print(f"{indent}准备返回：本层 n 仍为 {n}，结果 {result}")
    return result


assert factorial_trace(3, 0) == 6

### 7.6 漏写 return，内层算对也交不回来

错误版本确实调用了下一层，但没有返回结果。用最小输入 `1` 就能看到 `None`。
更深调用会尝试用数字乘 `None`，产生 `TypeError`。


In [ ]:
def factorial_missing_return(n):
    if n == 0:
        return 1
    n * factorial_missing_return(n - 1)  # 故意漏写 return


print(factorial_missing_return(1))  # None
assert factorial_missing_return(1) is None
try:
    factorial_missing_return(2)
except TypeError as error:
    print(type(error).__name__, str(error))
print("正确版本", factorial(2))  # 2

### 7.7 没有基例、参数不变会怎样？

以下错误写法只放在文本中，不执行，避免制造无限递归。

```python
def no_progress(n):
    return no_progress(n)  # 参数没变，也没有基例
```

递归前检查三个问题即可：哪个输入直接返回？下一层变成什么问题？什么量持续减小并最终到达基例？
Python 有递归深度限制；本讲用小输入理解机制。递归不保证比循环更快。


## 8. Exercises

以下单元格只保留注释，方便在其中填写代码，整本运行时不会遇到未完成函数。
建议自己完成后，再看下一节的结果和最后一节的实现。

### E1. 标签长度

实现 `label_lengths(data)`。保留长度至少为 3 的字符串，返回它们的长度。
先筛选类型，再调用 `len`，不要修改输入。

```text
['AI', 'Python', 7, 'art', ''] → [6, 3]
[] → []
```


In [ ]:
# 在这里定义 label_lengths(data)，再添加自己的调用。


### E2. 为不同班级生成加分规则

实现 `make_bonus(points)`，返回一个函数。返回的函数接收成绩，返回加分后的值。
不截断到 100 分；不要添加题目没有要求的规则。

```text
make_bonus(5)(80) → 85
make_bonus(0)(80) → 80
```


In [ ]:
# 在这里定义 make_bonus(points)。检查两种加分规则是否独立。


### E3. 按名单查成绩（Lec09 / Lec10 的 find_grades）

实现 `find_grades(grades, students)`。按名单顺序返回成绩列表。
假设姓名均存在；名单可为空、可以重复姓名。不得修改字典或名单。

```text
{'Ana': 'B', 'Matt': 'C', 'Katy': 'A'}, ['Matt', 'Katy', 'Matt']
→ ['C', 'A', 'C']
```


In [ ]:
# 在这里定义 find_grades(grades, students)。


### E4. 给错误版本补一个测试

回到 `absolute_buggy(x)`。写下一个能揭示错误的输入与期望值，
实现 `absolute_correct(x)`，再用负数、零、正数与原反例检查它。
解释为什么仅测试 `-2` 和 `2` 不够。


In [ ]:
# 在这里定义 absolute_correct(x)，并保留原反例。


### E5. 逐项相除（Lec09 p. 48）

实现 `pairwise_div(numerators, denominators)`。
两者是非空、等长数值列表，可用断言检查长度约定。
返回对应位置相除的新列表，不修改输入；分母出现零时，**抛出 ValueError**。
尝试捕获底层 `ZeroDivisionError`，再按接口约定重新抛出异常。

```text
[4, 5, 6], [1, 2, 3] → [4.0, 2.5, 2.0]
[4, 5, 6], [1, 0, 3] → ValueError
```


In [ ]:
# 在这里定义 pairwise_div(numerators, denominators)。


### E6. 查找字典的键，并统计键值相同的条目

1. 实现 `find_in_L(Ld, k)`，判断 `k` 是否为字典列表中至少一个字典的键。
   空列表返回 `False`；找到后可以立即返回 `True`。
2. 实现 `count_matches(d)`，统计键等于值的条目数。空字典返回 `0`。

```text
find_in_L([{1: 2}, {5: 25}], 25) → False
find_in_L([{1: 2}, {5: 25}], 5) → True
count_matches({1: 2, 'a': 'a', 5: 5}) → 2
```


In [ ]:
# 在这里定义 find_in_L(Ld, k) 和 count_matches(d)。


### E7. 汇总所有小测成绩

实现 `average_mq(data)`，计算嵌套字典中所有 `'mq'` 分数的算术平均值。
约定每位学生都存在 `'mq'` 列表，所有人合计至少有一条成绩；不修改输入。

```python
{'Ana': {'mq': [8, 10]}, 'Bob': {'mq': [6]}}
```

期望 `8.0`。解释 `7.5` 对应的是另一种什么统计方式。


In [ ]:
# 在这里定义 average_mq(data)。


### E8. 热门标签报表

复用 `generate_word_dict` 与 `occurs_often`，实现 `popular_tags(text, x)`。
只统计按空白分开的词，忽略大小写；返回次数严格大于 `x` 的频率组。
频率从大到小，同组词按首次出现顺序排列。

```text
'Python art python music ART python music', 1
→ [(['python'], 3), (['art', 'music'], 2)]
空文本 → []
```

另外检查 `occurs_often` 在阈值为 `0` 时能删空副本并正常返回，且原词频不变。


In [ ]:
# 在这里定义 popular_tags(text, x)，并添加边界检查。


### E9. 递归三题

分别实现 `power_solution(n, p)`、`factorial_solution(n)` 与 `sum_to(n)`。
`p`、`n` 的整数计数参数均为非负整数；本练习规定零次幂为 1。

先写下每题的基例、递推式、越来越小的量，再写代码。
其中 $S(0)=0$，$S(n)=n+S(n-1)$。

```text
power_solution(2, 3) → 8
factorial_solution(4) → 24
sum_to(5) → 15
```


In [ ]:
# 在这里定义 power_solution(n, p)、factorial_solution(n)、sum_to(n)。


## 9. Check Your Answers

| 题号 | 至少应检查的结果 |
|:---|:---|
| E1 | 混合示例为 `[6, 3]`；空列表为 `[]`；整数不会传给 `len` |
| E2 | 加 5 得 `85`，加 0 得 `80`；生成两个函数后仍分别有效 |
| E3 | 名单 `['Matt', 'Katy', 'Matt']` 得 `['C', 'A', 'C']`；空名单得 `[]` |
| E4 | `-1` 是反例，期望 `1`；两个分支都执行过不代表边界正确 |
| E5 | 正常为 `[4.0, 2.5, 2.0]`；零分母必须是 `ValueError` |
| E6 | 值为 25 不代表键有 25；匹配数为 `2`；空输入为 `False` / `0` |
| E7 | 按成绩条数加权是 `8.0`；每位学生等权是 `7.5` |
| E8 | 得到两组；阈值为 2 时只剩 python；空文本得 `[]` |
| E9 | 零次幂 `1`；`0! = 1`；空和 `0`；示例依次是 `8`、`24`、`15` |

如果结果不同，先缩小到一个反例，写清楚实际结果与期望结果，再决定打印哪些变量。


## 10. Reference Solutions

### E1–E2. 推导式与返回函数

E1 用连续的两个筛选条件，保证类型通过后才判断长度。
E2 返回内部函数对象，调用时再接收成绩。


In [ ]:
def label_lengths(data):
    return [len(item) for item in data if type(item) == str if len(item) >= 3]


def make_bonus(points):
    def add_bonus(score):
        return score + points

    return add_bonus


assert label_lengths(["AI", "Python", 7, "art", ""]) == [6, 3]
assert label_lengths([]) == []
bonus_five = make_bonus(5)
bonus_zero = make_bonus(0)
assert bonus_five(80) == 85
assert bonus_zero(80) == 80
assert bonus_five(90) == 95
print("E1–E2 通过")

### E3–E4. 按名单查询与边界回归

查字典无需先循环找键。绝对值的分界点应该是 `0`。


In [ ]:
def find_grades(grades, students):
    return [grades[name] for name in students]


def absolute_correct(x):
    if x < 0:
        return -x
    return x


grade_data = {"Ana": "B", "Matt": "C", "Katy": "A"}
requested_students = ["Matt", "Katy", "Matt"]
assert find_grades(grade_data, requested_students) == ["C", "A", "C"]
assert find_grades(grade_data, []) == []
assert grade_data == {"Ana": "B", "Matt": "C", "Katy": "A"}
assert requested_students == ["Matt", "Katy", "Matt"]
for number in [-2, -1, 0, 1, 2]:
    assert absolute_correct(number) == abs(number)
print("E3–E4 通过")

### E5. 按接口约定转换异常类型

长度由输入约定保证，断言帮助发现调用错误。
真正除到零时，由 `except ZeroDivisionError` 捕获，重新抛出题目要求的 `ValueError`。
`from error` 保留异常原因；这里不需要返回特殊数字代替错误。


In [ ]:
def pairwise_div(numerators, denominators):
    assert len(numerators) > 0
    assert len(numerators) == len(denominators)
    result = []
    for i in range(len(numerators)):
        try:
            quotient = numerators[i] / denominators[i]
        except ZeroDivisionError as error:
            raise ValueError("denominator contains zero") from error
        result.append(quotient)
    return result


numerators = [4, 5, 6]
denominators = [1, 2, 3]
assert pairwise_div(numerators, denominators) == [4.0, 2.5, 2.0]
assert pairwise_div([0], [2]) == [0.0]
assert numerators == [4, 5, 6] and denominators == [1, 2, 3]
for zeros in [[0, 2, 3], [1, 0, 3], [1, 2, 0]]:
    try:
        pairwise_div([4, 5, 6], zeros)
    except ValueError as error:
        print(type(error).__name__, str(error))
    else:
        raise AssertionError("零分母应当抛出 ValueError")
print("E5 通过")

### E6. 找到后返回，没找到时等循环结束

`return False` 必须在循环外，否则第一个字典没找到时就会提前结束。
统计时用 `items()` 同时访问键与值。


In [ ]:
def find_in_L(Ld, k):
    for dictionary in Ld:
        if k in dictionary:
            return True
    return False


def count_matches(d):
    count = 0
    for key, value in d.items():
        if key == value:
            count += 1
    return count


assert find_in_L([{1: 2}, {5: 25}], 5) is True
assert find_in_L([{1: 2}, {5: 25}], 25) is False
assert find_in_L([], 5) is False
assert count_matches({1: 2, "a": "a", 5: 5}) == 2
assert count_matches({}) == 0
print("E6 通过")

### E7–E8. 复用已经检查过的函数

`average_mq` 直接使用汇总函数，`popular_tags` 连接计数与分组。
这两个检查也属于集成测试。


In [ ]:
def average_mq(data):
    return get_average(data, "mq")


def popular_tags(text, x):
    counts = generate_word_dict(text)
    return occurs_often(counts, x)


mq_data = {"Ana": {"mq": [8, 10]}, "Bob": {"mq": [6]}}
mq_before = copy.deepcopy(mq_data)
assert average_mq(mq_data) == 8.0
assert mq_data == mq_before
assert average_mq({"Ana": {"mq": []}, "Bob": {"mq": [6]}}) == 6.0
example_text = "Python art python music ART python music"
assert popular_tags(example_text, 1) == [(["python"], 3), (["art", "music"], 2)]
assert popular_tags(example_text, 2) == [(["python"], 3)]
assert popular_tags("", 1) == []
counts = generate_word_dict(example_text)
counts_before = counts.copy()
assert occurs_often(counts, 0) == [(["python"], 3), (["art", "music"], 2)]
assert counts == counts_before
print("E7–E8 通过")

### E9. 三个递归函数，三套基例与组合规则

共同结构是「检查最小输入 → 调用更小问题 → 用本层数据组合并返回」。
幂与阶乘的基例返回 `1`；求和的基例返回 `0`，不要混淆。


In [ ]:
def power_solution(n, p):
    assert type(p) == int and p >= 0
    if p == 0:
        return 1
    return n * power_solution(n, p - 1)


def factorial_solution(n):
    assert type(n) == int and n >= 0
    if n == 0:
        return 1
    return n * factorial_solution(n - 1)


def sum_to(n):
    assert type(n) == int and n >= 0
    if n == 0:
        return 0
    return n + sum_to(n - 1)


assert power_solution(2, 0) == 1
assert power_solution(2, 3) == 8
assert power_solution(0, 0) == 1
assert factorial_solution(0) == 1
assert factorial_solution(4) == 24
assert sum_to(0) == 0
assert sum_to(1) == 1
assert sum_to(5) == 15
for n in range(7):
    assert sum_to(n) == n * (n + 1) // 2
print("E9 通过")

## 离开教室前

试着不用代码回答，再回看相应例子验证。

1. 列表推导式什么时候执行结果表达式？
2. `return cost` 和 `return cost(3)` 各返回什么？
3. 为什么走过两个分支，还需要边界用例？
4. 异常被处理后，程序从哪里继续？
5. `in` 检查字典的什么？`copy()` 是否复制了内部列表？
6. 为什么汇总小测不能把每个人的列表直接 `append` 后求和？
7. 一层递归正在等什么？参数如何确保靠近基例？

所有演示与参考实现都可按顺序运行。再次运行前可以重启内核，检查是否依赖遗留变量。
